# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook guides you through loading and exploring the [FAIR^2](https://doi.org/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL. The schema fully describes available tables (record sets), their fields, and how to access the records.

In [ ]:
# Ensure `mlcroissant` is installed (restart the runtime if asked)
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s as described by the Croissant schema. This helps us know what structured data (data tables) are available.

In [ ]:
# Examine the available record sets from metadata
record_sets = metadata.record_sets

print(f"Number of record sets: {len(record_sets)}")
for idx, rec in enumerate(record_sets):
    print(f"[{idx}] @id: {rec.id}\n    name: {rec.name} \n    description: {rec.description}\n")

# List fields for each record set
for rec in record_sets:
    print(f"Fields for record set '{rec.name}' (id: {rec.id}):")
    for field in rec.fields:
        print(f"    - @id: {field.id} | name: {field.name} | dataType: {field.data_type}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview section.

In [ ]:
# For demonstration purposes, pick the first record set.
record_set_ids = [rec.id for rec in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    print(f"Loading records for record set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  DataFrame shape: {df.shape}\n")

# Show columns from first record set
main_record_set = record_set_ids[0] if record_set_ids else None
if main_record_set is not None and not dataframes[main_record_set].empty:
    print("Available columns:")
    print(dataframes[main_record_set].columns.tolist())
    display(dataframes[main_record_set].head())
else:
    print("No records found in the primary record set.")

## 4. Exploratory Data Analysis (EDA)
Apply data processing to a numeric field of interest, such as filtering records, normalizing values, or grouping by a key attribute.

In [ ]:
# Pick a numeric field from the main record set
df = dataframes[main_record_set].copy()

# Display all available field names for user's reference
print("Available columns:", df.columns.tolist())

# Example: Try common mass spectrometry fields based on dataset description
# Let's guess a column for peptide counts or MW (molecular weight),
# common possibilities: 'Peptide_Count', 'MW', 'Abundance', etc.
possible_numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]
if not possible_numeric_fields:
    # Try to coerce columns to numeric if possible
    for col in df.columns:
        try:
            df[col] = pd.to_numeric(df[col])
        except Exception:
            continue
    possible_numeric_fields = [col for col in df.columns if df[col].dtype in ['float64', 'int64']]

if possible_numeric_fields:
    numeric_field = possible_numeric_fields[0]
else:
    raise ValueError("No numeric fields detected in the record set.")
print(f"Using numeric field: {numeric_field}")

# Filtering records with values above an example threshold
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} (z-score normalization):")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by an obvious string/categorical field
possible_group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and col != numeric_field]
group_field = possible_group_fields[0] if possible_group_fields else None
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by '{group_field}':\n", grouped_df.head())

## 5. Visualization
Visualize distributions or relationships between selected fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the selected numeric field
plt.figure(figsize=(8,5))
sns.histplot(df[numeric_field], kde=True, bins=30)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If there is another numeric field, visualize their relationship
if len(possible_numeric_fields) > 1:
    second_numeric = possible_numeric_fields[1]
    plt.figure(figsize=(7,5))
    sns.scatterplot(data=df, x=numeric_field, y=second_numeric)
    plt.title(f"Scatter between {numeric_field} and {second_numeric}")
    plt.xlabel(numeric_field)
    plt.ylabel(second_numeric)
    plt.show()

## 6. Conclusion

In this notebook, you've loaded and explored the FAIR⁲ dataset using `mlcroissant`. You reviewed available record sets and fields, extracted records, applied filtering and normalization to a numeric field, optionally grouped by a category, and visualized distributions and relationships.

This workflow can be adapted for deeper protein analysis, including more advanced filtering, additional visualizations, or machine learning applications, tailored to the dataset's domain.

For more about the dataset, visit the [FAIR^2 dataset page](https://doi.org/10.71728/senscience.vq4a-28xa) or consult the Croissant schema metadata.